# Object Tracking (OpenCV Bootcamp NB11)

Track a race car through a video with one of OpenCV's single-object trackers, write an annotated
video, and play it back inline.

Modernized for OpenCV 4.10+ / 5.x and NumPy 2.x. Run the cells top to bottom (**Run All** works).

**Install:** `pip install -r requirements.txt`
Use `opencv-contrib-python`, **not** `opencv-python` — KCF, CSRT and the legacy trackers live in contrib.
Don't install both packages at once; they conflict.

## 1. Imports

In [ ]:
import os
import shutil
import subprocess
from zipfile import ZipFile
from urllib.request import urlretrieve

import cv2
import matplotlib.pyplot as plt
from IPython.display import Video, display

print("OpenCV", cv2.__version__)

## 2. Configuration

Change `TRACKER_TYPE` to try a different tracker, then re-run from this cell down.

| Tracker | Notes |
|---|---|
| `MIL`, `KCF`, `CSRT` | Classic trackers, work out of the box |
| `BOOSTING`, `TLD`, `MEDIANFLOW`, `MOSSE` | Legacy API (`cv2.legacy`) |
| `NANO`, `VIT`, `DASIAMRPN` | Deep-learning trackers — need `.onnx` model files in `models/` |
| `GOTURN` | Removed in OpenCV 5; only available on 4.x with model files in `models/` |

In [ ]:
TRACKER_TYPE = "CSRT"

tracker_types = ["BOOSTING", "MIL", "KCF", "CSRT", "TLD", "MEDIANFLOW",
                 "GOTURN", "MOSSE", "NANO", "VIT", "DASIAMRPN"]
assert TRACKER_TYPE in tracker_types, f"Unknown tracker type: {TRACKER_TYPE}"

DATA_DIR = "data"        # downloaded video assets
MODELS_DIR = "models"    # .onnx / caffe files for the deep-learning trackers
OUTPUT_DIR = "outputs"   # annotated videos written here
for d in (DATA_DIR, MODELS_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

ASSETS_URL = "https://www.dropbox.com/s/ld535c8e0vueq6x/opencv_bootcamp_assets_NB11.zip?dl=1"
video_input_file_name = os.path.join(DATA_DIR, "race_car.mp4")

# Initial bounding box (x, y, w, h) around the car in the first frame
bbox = (1300, 405, 160, 120)

## 3. Download assets

Downloads and unzips `race_car.mp4` (plus the GOTURN model files) into `data/`. Skipped if the video is already there.

In [ ]:
def download_and_unzip(url, save_path):
    print("Downloading and extracting assets....", end="")
    urlretrieve(url, save_path)
    try:
        with ZipFile(save_path) as z:
            z.extractall(os.path.dirname(save_path))
        print("Done")
    except Exception as e:
        print("\nInvalid file.", e)
    finally:
        os.remove(save_path)

if not os.path.exists(video_input_file_name):
    download_and_unzip(ASSETS_URL, os.path.join(DATA_DIR, "opencv_bootcamp_assets_NB11.zip"))
else:
    print("Assets already present:", video_input_file_name)

## 4. Drawing helpers

In [ ]:
def drawRectangle(frame, bbox):
    x, y, w, h = map(int, bbox)
    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2, cv2.LINE_AA)

def displayRectangle(frame, bbox):
    plt.figure(figsize=(20, 10))
    frameCopy = frame.copy()
    drawRectangle(frameCopy, bbox)
    plt.imshow(cv2.cvtColor(frameCopy, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

def drawText(frame, txt, location, color=(50, 170, 50)):
    cv2.putText(frame, txt, location, cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3, cv2.LINE_AA)

## 5. Tracker factory

In [ ]:
def _require(module, name):
    fn = getattr(module, name, None)
    if fn is None:
        raise RuntimeError(
            f"{name} is not available in this OpenCV build ({cv2.__version__}). "
            "Install opencv-contrib-python (and uninstall opencv-python)."
        )
    return fn

def _model(filename):
    path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model file missing: {path}")
    return path

def create_tracker(tracker_type):
    legacy = getattr(cv2, "legacy", None)

    if tracker_type == "MIL":
        return cv2.TrackerMIL_create()
    if tracker_type == "KCF":
        return _require(cv2, "TrackerKCF_create")()
    if tracker_type == "CSRT":
        return _require(cv2, "TrackerCSRT_create")()

    # Legacy trackers (contrib only, older API)
    if tracker_type in ("BOOSTING", "TLD", "MEDIANFLOW", "MOSSE"):
        if legacy is None:
            raise RuntimeError("cv2.legacy missing — install opencv-contrib-python.")
        name = {"BOOSTING": "TrackerBoosting_create", "TLD": "TrackerTLD_create",
                "MEDIANFLOW": "TrackerMedianFlow_create", "MOSSE": "TrackerMOSSE_create"}[tracker_type]
        return _require(legacy, name)()

    # Deep-learning trackers — each needs model files in MODELS_DIR
    if tracker_type == "GOTURN":   # removed in OpenCV 5; reads goturn.prototxt + goturn.caffemodel from the cwd
        return _require(cv2, "TrackerGOTURN_create")()
    if tracker_type == "NANO":
        p = cv2.TrackerNano_Params()
        p.backbone = _model("nanotrack_backbone_sim.onnx")
        p.neckhead = _model("nanotrack_head_sim.onnx")
        return cv2.TrackerNano_create(p)
    if tracker_type == "VIT":
        p = cv2.TrackerVit_Params()
        p.net = _model("object_tracking_vittrack_2023sep.onnx")
        return cv2.TrackerVit_create(p)
    if tracker_type == "DASIAMRPN":
        p = cv2.TrackerDaSiamRPN_Params()
        p.model = _model("dasiamrpn_model.onnx")
        p.kernel_r1 = _model("dasiamrpn_kernel_r1.onnx")
        p.kernel_cls1 = _model("dasiamrpn_kernel_cls1.onnx")
        return cv2.TrackerDaSiamRPN_create(p)

    raise ValueError(f"Unknown tracker type: {tracker_type}")

## 6. Open the video and initialise the tracker

Shows the first frame with the starting bounding box. To pick the box by hand instead, uncomment the
`selectROI` line (opens a desktop window; doesn't work in Colab).

In [ ]:
video = cv2.VideoCapture(video_input_file_name)
if not video.isOpened():
    raise IOError(f"Could not open video: {video_input_file_name}")

ok, frame = video.read()
if not ok:
    raise IOError("Could not read first frame")

width  = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
src_fps = video.get(cv2.CAP_PROP_FPS) or 30
print(f"{width}x{height} @ {src_fps:.1f} fps")

# bbox = cv2.selectROI("Select object", frame, showCrosshair=False); cv2.destroyAllWindows()
displayRectangle(frame, bbox)

tracker = create_tracker(TRACKER_TYPE)
tracker.init(frame, tuple(int(v) for v in bbox))   # new Tracker API wants an integer Rect

## 7. Run the tracker

Tracks the object frame by frame and writes an annotated video to `outputs/`. CSRT takes a minute or two.

In [ ]:
video_output_file_name = os.path.join(OUTPUT_DIR, f"race_car-{TRACKER_TYPE}.mp4")
video_out = cv2.VideoWriter(video_output_file_name,
                            cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (width, height))

n_frames = n_lost = 0
while True:
    ok, frame = video.read()
    if not ok:
        break

    timer = cv2.getTickCount()
    ok, box = tracker.update(frame)
    fps = cv2.getTickFrequency() / (cv2.getTickCount() - timer)

    if ok:
        drawRectangle(frame, box)
    else:
        n_lost += 1
        drawText(frame, "Tracking failure detected", (80, 140), (0, 0, 255))

    drawText(frame, f"{TRACKER_TYPE} Tracker", (80, 60))
    drawText(frame, f"FPS : {int(fps)}", (80, 100))
    video_out.write(frame)
    n_frames += 1

video.release()
video_out.release()
print(f"Tracked {n_frames} frames ({n_lost} lost). Wrote {video_output_file_name}")

## 8. Convert to H.264 and play inline

Browsers can't play the `mp4v` codec, so this re-encodes with `ffmpeg` (`brew install ffmpeg`) for inline playback.

In [ ]:
h264_name = os.path.join(OUTPUT_DIR, f"race_car_track_{TRACKER_TYPE}_x264.mp4")
if shutil.which("ffmpeg"):
    subprocess.run(["ffmpeg", "-y", "-i", video_output_file_name, "-c:v", "libx264",
                    "-pix_fmt", "yuv420p", h264_name, "-hide_banner", "-loglevel", "error"],
                   check=True)
    print("Wrote", h264_name)
    display(Video(h264_name, embed=True, width=1024))
else:
    print("ffmpeg not found — skipping H.264 conversion (brew install ffmpeg / apt install ffmpeg)")